# Polar Coordinate System

The polar coordinate system is used for pie charts and polar plots.

Note that the polar coordinate system may change the default expansion for position scales:

- Discrete x-scale is expanded so that the first and last values don't overlap.
- Continuous position scales have their expands set to zero so that the pie chart doesn't get an inner hole, and the first and last sectors don't get a gap.

In [1]:
%useLatestDescriptors
%use lets-plot
%use dataframe

In [2]:
LetsPlot.getInfo()

Lets-Plot Kotlin API v.4.7.2. Frontend: Notebook with dynamically loaded JS. Lets-Plot JS v.4.3.2.

In [3]:
LetsPlot.theme = themeGrey()

In [4]:
val mpgDf = DataFrame.readCSV("https://raw.githubusercontent.com/JetBrains/lets-plot-docs/master/data/mpg.csv")
mpgDf.head()

untitled,manufacturer,model,displ,year,cyl,trans,drv,cty,hwy,fl,class
1,audi,a4,1.800000,1999,4,auto(l5),f,18,29,p,compact
2,audi,a4,1.800000,1999,4,manual(m5),f,21,29,p,compact
3,audi,a4,2.000000,2008,4,manual(m6),f,20,31,p,compact
4,audi,a4,2.000000,2008,4,auto(av),f,21,30,p,compact
5,audi,a4,2.800000,1999,6,auto(l5),f,16,26,p,compact


In [5]:
val mpgData = mpgDf.toMap()

## Pie Chart

`geomBar()` with `coordPolar()` get transformed into a pie chart. Note the `theta = "y"` parameter.

In [6]:
val bars = letsPlot(mpgData) + geomBar(size = 0) { fill = asDiscrete("cyl") }

gggrid(listOf(
    bars + ggtitle("From Bar Chart"),
    bars + coordPolar(theta = "y") + ggtitle("To Pie Chart")
))

## Various Bar Charts in the Polar Coordinate System

In [7]:
val bullEye = letsPlot(mpgData) + 
    geomBar(size = 0, showLegend = false) { fill = asDiscrete("cyl") }
    
val radialBar = letsPlot(mpgData) + 
    geomBar(size = 0, position = positionDodge(), showLegend = false) { fill = asDiscrete("cyl") }
    
val coxcomb = letsPlot(mpgData) + 
    geomBar(size = 0, position = positionDodge(), showLegend = false) { fill = asDiscrete("cyl") }


gggrid(
    listOf(
        bullEye + coordPolar() + ggtitle("Bulls eye"),
        radialBar + coordPolar(theta = "y") + ggtitle("Radial bar chart"),
        coxcomb + coordPolar() + ggtitle("Coxcomb plot"),

        bullEye + ggtitle("was:"),
        radialBar + ggtitle("was:"),
        coxcomb + ggtitle("was:"),
    ), 
    ncol = 3
)

## Radar Plot
Use the `flat` parameter in path-based geoms to construct a radar plot.

In [8]:
val studentData = mapOf(
    "subj" to listOf("progr", "math", "physic", "chemistry", "biology"),
    "subjId" to listOf(1, 2, 3, 4, 5),
    "student" to List(5) { "John" },
    "score" to listOf(19, 15, 18, 12, 9)
)
val labels = mapOf(1 to "progr", 2 to "math", 3 to "physic", 4 to "chemistry", 5 to "biology")

letsPlot(studentData) +
    geomArea(flat = true) {   // <-- flat. I.e., do not transform segments to curves
        x = "subjId"; y = "score"
    } +
    geomPoint() { x = "subjId"; y = "score" } +
    scaleXDiscrete(labels = labels) + 
    coordPolar()

## Heatmap Plot

In [9]:
val maxR = 100
val stepsR = 4 * maxR
val maxTheta = 2 * PI
val stepsTheta = 200
val dataFunction: (Double, Double) -> Double = { x, y -> sin(x * 7) + cos(y / 11) }

In [10]:
fun linspace(start: Double, stop: Double, num: Int): List<Double> {
    if (num <= 0) return emptyList()
    if (num == 1) return listOf(start)
    val step = (stop - start) / (num - 1)
    return List(num) { start + it * step }
}

fun simpleMeshgrid(xs: List<Double>, ys: List<Double>): Pair<List<List<Double>>, List<List<Double>>> {
    return Pair(
        List(ys.size) { xs },
        ys.map { y -> List(xs.size) { y } }
    )
}

fun getData(xs: List<Double>, ys: List<Double>, f: (Double, Double) -> Double): Map<String, List<Double>> {
    val zs = (xs zip ys).map { p -> f(p.first, p.second) }
    return mapOf("x" to xs, "y" to ys, "z" to zs)
}

In [11]:
val (gridR, gridTheta) = simpleMeshgrid(
    linspace(0.0, maxR.toDouble(), stepsR),
    linspace(0.0, maxTheta.toDouble(), stepsTheta)
)

val dataMap = getData(gridTheta.flatten(), gridR.flatten(), dataFunction)

In [12]:
val p = ggplot(dataMap) +
    geomTile(size = 1, tooltips = tooltipsNone) { x = "x"; y = "y"; color = "z"; fill = "z" } +
    scaleBrewer(listOf("color", "fill"), palette = "Spectral", direction = -1) +
    theme(axisTitle = elementBlank())

gggrid(listOf(
    p + coordCartesian() + ggtitle("Rectangular Heatmap"),
    p + coordPolar() + ggtitle("Polar Heatmap"),
))

## Theming

The polar coordinate system supports axis theme options, incorporates specialized logic for the theme's `panelInset` parameter, and offers its own `transformBkgr` option to manage plot panel transformation:

In [13]:
letsPlot(mpgData) + 
    geomBar(stat = Stat.identity, position = positionDodge()) { x = "model"; y = "cty"; fill = "cty" } +
    scaleFillGradient(low = "red", high = "white", limits = 5 to 40) +
    theme(
        axisLineY = elementLine(color = "red", size = 2),
        axisLineX = elementLine(color = "blue", size = 2),
        axisTicksLengthY = 5,
        axisTicksLengthX = 10,
        axisTicksY = elementLine(size = 5, color = "red"), 
        axisTicksX = elementLine(size = 3, color = "blue"),
        axisTextX = elementText(color = "blue", angle = 10),
        axisTextY = elementText(color = "red"),
        panelInset = listOf(20, 140, 30, 135)                // New! Expand the panel to fit axis labels.
    ) + 
    ggsize(900, 500) + 
    coordPolar(transformBkgr = false)                        // Keep the old school rectangular background.